# ImageCLEF 2026 — Multimodal Reasoning Pipeline
## Fine-Tuned Thinking VLMs with Self-Consistency for Multilingual Exam Reasoning

### Pipeline
1. Download EXAMS-V (training) + Competition test datasets
2. Zero-shot baseline (Qwen3-VL-8B-Thinking)
3. Prompt engineering on 500-sample subset
4. QLoRA fine-tuning on 16K training samples
5. Self-consistency k=3 on competition test sets
6. OpenQA inference
7. Package submissions

**Models:** Qwen3-VL-8B-Thinking (Normal) | Qwen3-VL-4B-Thinking (Tiny)  
**GPU Budget:** ~30h on H200  
**Deadline:** May 10, 2026  
**Target:** 85%+ MCQ accuracy (2025 winner: 81.4%)

### Datasets
| Purpose | Dataset | Samples |
|---------|---------|---------|
| **Training** (with labels) | `MBZUAI/EXAMS-V` | 16,300 train / 4,650 val |
| **Test: Visual MCQ** | `SU-FMI-AI/ImageCLEF-MR2026-MCQ-Visual` | 1,117 |
| **Test: Text MCQ** | `SU-FMI-AI/ImageCLEF-MR2026-MCQ-Textual` | 1,653 |
| **Test: Visual OpenQA** | `SU-FMI-AI/ImageCLEF-MR2026-OpenQA-Visual` | 439 |
| **Test: Text OpenQA** | `SU-FMI-AI/ImageCLEF-MR2026-OpenQA-Textual` | 424 |

**Important:** EXAMS-V is for training ONLY. Competition test data comes from separate datasets with NO answer keys.

---
## 0. Setup

In [ ]:
!pip install -q --user datasets transformers>=4.57.0 accelerate peft bitsandbytes trl pillow qwen-vl-utils torch torchvision
!pip install -q --user vllm>=0.11.0 openai

In [ ]:
import torch
print(f"torch {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.0f} GB")

In [ ]:
import json
import os
import time
import random
import base64
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path
from PIL import Image

import numpy as np
from tqdm.notebook import tqdm

# ── Configuration ──────────────────────────────────────────────
MODEL_8B = "Qwen/Qwen3-VL-8B-Thinking"   # Normal category (>=8B)
MODEL_4B = "Qwen/Qwen3-VL-4B-Thinking"   # Tiny category (<=7B)

# ── Persistent Storage ─────────────────────────────────────────
VAULT_DIR = Path("/home/jovyan/vault/CLEF/ImageCLEF")

if VAULT_DIR.exists():
    DATA_DIR = VAULT_DIR / "data"
    OUTPUT_DIR = VAULT_DIR / "outputs"
    HF_CACHE = VAULT_DIR / "hf_cache"
    HF_CACHE.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(HF_CACHE)
    print(f"Persistent vault: {VAULT_DIR}")
else:
    DATA_DIR = Path("./data")
    OUTPUT_DIR = Path("./outputs")
    print("Using local storage (set VAULT_DIR for persistence)")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Test Mode ──────────────────────────────────────────────────
TEST_MODE = True   # Set False for full run
TEST_LIMIT = 20

print(f"Data: {DATA_DIR}")
print(f"Outputs: {OUTPUT_DIR}")
print(f"Test mode: {TEST_MODE} (limit={TEST_LIMIT})")

---
## 1. Download Data

In [ ]:
from datasets import load_dataset

def download_exams_v():
    """Download EXAMS-V for TRAINING and VALIDATION only (has answer labels)."""
    print("Downloading EXAMS-V (training data)...")
    ds = load_dataset("MBZUAI/EXAMS-V")

    for split in ["train", "validation"]:
        split_dir = DATA_DIR / "exams_v" / split / "images"
        split_dir.mkdir(parents=True, exist_ok=True)
        meta_path = DATA_DIR / "exams_v" / f"{split}_metadata.json"

        if meta_path.exists():
            with open(meta_path) as f:
                existing = json.load(f)
            print(f"  {split}: already downloaded ({len(existing)} samples)")
            continue

        metadata = []
        for item in tqdm(ds[split], desc=split):
            sid = item["sample_id"]
            img_path = split_dir / f"{sid}.png"
            if not img_path.exists():
                item["image"].save(img_path)
            metadata.append({
                "id": sid,
                "answer_key": item["answer_key"],
                "type": item["type"],
                "subject": item["subject"],
                "language": item["language"],
            })

        with open(meta_path, "w") as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
        print(f"  {split}: {len(metadata)} samples")

    print("EXAMS-V done.\n")


def download_competition_test():
    """Download the 4 SEPARATE competition test datasets (NO answer keys)."""
    COMPETITION_DATASETS = {
        "mcq_visual":    "SU-FMI-AI/ImageCLEF-MR2026-MCQ-Visual",
        "mcq_textual":   "SU-FMI-AI/ImageCLEF-MR2026-MCQ-Textual",
        "openqa_visual":  "SU-FMI-AI/ImageCLEF-MR2026-OpenQA-Visual",
        "openqa_textual": "SU-FMI-AI/ImageCLEF-MR2026-OpenQA-Textual",
    }

    for name, hf_id in COMPETITION_DATASETS.items():
        print(f"Downloading {name}...")
        ds = load_dataset(hf_id)

        for split in ds.keys():
            split_dir = DATA_DIR / name / split / "images"
            split_dir.mkdir(parents=True, exist_ok=True)
            meta_path = DATA_DIR / name / f"{split}_metadata.json"

            if meta_path.exists():
                with open(meta_path) as f:
                    existing = json.load(f)
                print(f"  {split}: already downloaded ({len(existing)})")
                continue

            metadata = []
            for item in tqdm(ds[split], desc=f"{name}/{split}"):
                qid = item["question_id"]
                if "image" in item and item["image"] is not None:
                    img_path = split_dir / f"{qid}.png"
                    if not img_path.exists():
                        item["image"].save(img_path)
                entry = {
                    "question_id": qid,
                    "type": item.get("type", ""),
                    "subject": item.get("subject", ""),
                    "language": item.get("language", ""),
                }
                # OpenQA train/dev splits have answers; test splits do NOT
                if "answer" in item and item["answer"]:
                    entry["answer"] = item["answer"]
                if "answers" in item and item["answers"]:
                    entry["answers"] = item["answers"]
                metadata.append(entry)

            with open(meta_path, "w") as f:
                json.dump(metadata, f, indent=2, ensure_ascii=False)
            print(f"  {split}: {len(metadata)} samples")
        print()


download_exams_v()
download_competition_test()
print("All data downloaded!")

In [ ]:
# ── Data Summary ──────────────────────────────────────────────
print("TRAINING DATA (with answer labels):")
for ds_name in ["exams_v"]:
    ds_dir = DATA_DIR / ds_name
    if not ds_dir.exists():
        continue
    for meta_file in sorted(ds_dir.glob("*_metadata.json")):
        with open(meta_file) as f:
            data = json.load(f)
        split = meta_file.stem.replace("_metadata", "")
        langs = defaultdict(int)
        for item in data:
            langs[item.get("language", "?")] += 1
        top = sorted(langs.items(), key=lambda x: -x[1])[:5]
        print(f"  {ds_name}/{split}: {len(data)} samples — {', '.join(f'{l}({n})' for l, n in top)}")

print("\nCOMPETITION TEST DATA (NO answer labels — submit to leaderboard):")
for ds_name in ["mcq_visual", "mcq_textual", "openqa_visual", "openqa_textual"]:
    ds_dir = DATA_DIR / ds_name
    if not ds_dir.exists():
        continue
    for meta_file in sorted(ds_dir.glob("*_metadata.json")):
        with open(meta_file) as f:
            data = json.load(f)
        split = meta_file.stem.replace("_metadata", "")
        langs = defaultdict(int)
        for item in data:
            langs[item.get("language", "?")] += 1
        top = sorted(langs.items(), key=lambda x: -x[1])[:6]
        print(f"  {ds_name}/{split}: {len(data)} samples — {', '.join(f'{l}({n})' for l, n in top)}")

---
## 2. Load Model

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

ACTIVE_MODEL = MODEL_8B   # Switch to MODEL_4B for Tiny

print(f"Loading {ACTIVE_MODEL}...")
processor = AutoProcessor.from_pretrained(ACTIVE_MODEL, trust_remote_code=True)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    ACTIVE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation="sdpa",
)
model.eval()

print(f"Model loaded: {ACTIVE_MODEL}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

---
## 3. Inference Engine

In [ ]:
PROMPTS = {
    "direct": (
        "Look at this exam question image. Identify the question and all answer options. "
        "Select the correct answer. Reply with ONLY the letter (A, B, C, D, or E)."
    ),
    "thinking": (
        "Look at this exam question image carefully.\n"
        "1. Read the question and ALL answer options.\n"
        "2. If there are diagrams, graphs, tables, or visual elements, analyze them.\n"
        "3. Think step by step about the correct answer.\n"
        "4. Select exactly ONE answer.\n\n"
        "Answer with a single letter: A, B, C, D, or E."
    ),
    "decomposed": (
        "Look at this exam question image.\n"
        "First, describe what you see: the question text, answer options, and any "
        "visual elements (diagrams, graphs, tables, chemical structures).\n"
        "Then, reason step by step to determine the correct answer.\n"
        "Finally, state your answer as a single letter: A, B, C, D, or E."
    ),
}

# ── Language-Matched Prompting ─────────────────────────────────
# Qwen3-VL performs better when prompt language matches question language.
LANG_PROMPTS = {
    "English": "Answer with a single letter: A, B, C, D, or E.",
    "Chinese": "用一个字母回答：A、B、C、D 或 E。",
    "Bulgarian": "Отговорете само с една буква: A, B, C, D или E.",
    "Croatian": "Odgovorite samo jednim slovom: A, B, C, D ili E.",
    "Serbian": "Одговорите само једним словом: A, B, C, D или E.",
    "Italian": "Rispondi con una sola lettera: A, B, C, D o E.",
}

# ── Subject-Aware Prefix ──────────────────────────────────────
def get_subject_prefix(subject: str) -> str:
    """Generate subject-aware prefix. Zero cost — just string formatting."""
    if subject and subject != "unknown":
        return f"You are an expert in {subject}. "
    return ""


def build_prompt(base_prompt: str, subject: str = "", language: str = "") -> str:
    """Build final prompt with subject-aware prefix and language-matched answer line."""
    prompt = get_subject_prefix(subject) + base_prompt
    # Replace the English answer instruction with language-matched version
    if language in LANG_PROMPTS and language != "English":
        prompt = prompt.replace(
            "Answer with a single letter: A, B, C, D, or E.",
            LANG_PROMPTS[language]
        )
    return prompt


def predict_mcq(image_path: str, prompt_text: str, do_sample=False, temperature=1.0):
    """Predict MCQ answer for a single image."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{image_path}"},
            {"type": "text", "text": prompt_text},
        ],
    }]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text],
        images=[Image.open(image_path).convert("RGB")],
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512 if "thinking" in prompt_text.lower() or "step by step" in prompt_text.lower() else 50,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = processor.decode(new_tokens, skip_special_tokens=True).strip()

    answer = extract_answer(response)
    return answer, response


def extract_answer(response: str) -> str:
    """Extract A/B/C/D/E from model response."""
    response = response.strip()

    if response.upper() in {"A", "B", "C", "D", "E"}:
        return response.upper()

    import re
    patterns = [
        r'(?:answer|Answer|ANSWER)\s*(?:is|:)?\s*([A-E])',
        r'\b([A-E])\s*(?:\.|\)|$)',
        r'\*\*([A-E])\*\*',
    ]
    for pattern in patterns:
        match = re.search(pattern, response)
        if match:
            return match.group(1).upper()

    for ch in reversed(response):
        if ch.upper() in {"A", "B", "C", "D", "E"}:
            return ch.upper()

    return "A"


# ── Quick test ──────────────────────────────────────────────────
val_meta = DATA_DIR / "exams_v" / "validation_metadata.json"
with open(val_meta) as f:
    val_data = json.load(f)

test_item = val_data[0]
test_img = str(DATA_DIR / "exams_v" / "validation" / "images" / f"{test_item['id']}.png")
prompt = build_prompt(PROMPTS["thinking"], test_item.get("subject", ""), test_item.get("language", ""))
answer, response = predict_mcq(test_img, prompt)
print(f"Question: {test_item['id']} ({test_item['language']}, {test_item['subject']})")
print(f"Prompt preview: {prompt[:100]}...")
print(f"Predicted: {answer} | Gold: {test_item['answer_key']} | {'CORRECT' if answer == test_item['answer_key'] else 'WRONG'}")
print(f"Response: {response[:200]}...")

---
## 4. Zero-Shot Baseline + Prompt Engineering

Run 3 prompt variants on a 500-sample stratified subset to find the best prompt.

In [ ]:
# ── Create 500-sample stratified val subset ────────────────────
def create_val_subset(n=500, seed=42):
    subset_path = DATA_DIR / "exams_v" / "val_subset_500.json"
    if subset_path.exists():
        with open(subset_path) as f:
            subset = json.load(f)
        print(f"Loaded existing subset: {len(subset)} samples")
        return subset

    with open(DATA_DIR / "exams_v" / "validation_metadata.json") as f:
        val_data = json.load(f)

    random.seed(seed)
    by_lang = defaultdict(list)
    for item in val_data:
        by_lang[item["language"]].append(item)

    per_lang = max(1, n // len(by_lang))
    subset = []
    for lang, items in by_lang.items():
        random.shuffle(items)
        subset.extend(items[:per_lang])

    random.shuffle(subset)
    subset = subset[:n]

    with open(subset_path, "w") as f:
        json.dump(subset, f, indent=2)
    print(f"Created subset: {len(subset)} samples, {len(by_lang)} languages")
    return subset


val_subset = create_val_subset()

In [ ]:
def run_evaluation(data, split_name, prompt_name, prompt_text, experiment_name,
                   do_sample=False, temperature=1.0):
    """Run evaluation on a list of items. Uses subject-aware + language-matched prompting."""
    exp_dir = OUTPUT_DIR / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    results_path = exp_dir / "predictions.json"

    # Resume
    results = []
    done_ids = set()
    if results_path.exists():
        with open(results_path) as f:
            results = json.load(f)
        done_ids = {r["id"] for r in results}

    if split_name == "validation":
        image_dir = DATA_DIR / "exams_v" / "validation" / "images"
    else:
        image_dir = DATA_DIR / "exams_v" / split_name / "images"

    items = data if TEST_MODE is False else data[:TEST_LIMIT]
    remaining = [it for it in items if it["id"] not in done_ids]

    if not remaining:
        print(f"  {experiment_name}: already done ({len(done_ids)})")
    else:
        print(f"  {experiment_name}: {len(remaining)} remaining of {len(items)}")

    t0 = time.time()
    for i, item in enumerate(tqdm(remaining, desc=experiment_name)):
        img_path = str(image_dir / f"{item['id']}.png")
        if not os.path.exists(img_path):
            continue

        # Build prompt with subject-aware prefix + language-matched answer line
        full_prompt = build_prompt(prompt_text, item.get("subject", ""), item.get("language", ""))
        answer, response = predict_mcq(img_path, full_prompt, do_sample, temperature)
        is_correct = answer == item["answer_key"]

        results.append({
            "id": item["id"],
            "answer_key": answer,
            "gold": item["answer_key"],
            "correct": is_correct,
            "language": item.get("language", ""),
            "subject": item.get("subject", ""),
            "type": item.get("type", ""),
            "response": response[:300],
        })

        # Checkpoint every 50
        if (i + 1) % 50 == 0:
            with open(results_path, "w") as f:
                json.dump(results, f, indent=2)

    # Final save
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)

    # Analysis
    correct = sum(1 for r in results if r.get("correct"))
    total = len(results)
    acc = correct / total * 100 if total else 0
    print(f"  {experiment_name}: {correct}/{total} = {acc:.2f}%")

    return results, acc

In [ ]:
# ── Prompt Ablation on 500-sample subset ──────────────────────
print("Prompt Engineering Ablation")
print("=" * 55)

prompt_results = {}
for prompt_name, prompt_text in PROMPTS.items():
    _, acc = run_evaluation(
        val_subset, "validation", prompt_name, prompt_text,
        experiment_name=f"prompt_{prompt_name}"
    )
    prompt_results[prompt_name] = acc

print("\nResults:")
for name, acc in sorted(prompt_results.items(), key=lambda x: -x[1]):
    print(f"  {name:<15} {acc:.2f}%")

BEST_PROMPT = max(prompt_results, key=prompt_results.get)
print(f"\nBest prompt: {BEST_PROMPT} ({prompt_results[BEST_PROMPT]:.2f}%)")

---
## 5. [OPTIONAL] QLoRA Fine-Tuning

Fine-tune on 16K EXAMS-V training samples. ~9h for 8B, ~5h for 4B on H200.

**Skip this section if running zero-shot only.** Jump to Section 7 for test inference.
Sections 5-6 can be run later if you decide to fine-tune.

In [ ]:
# ── Free GPU memory before fine-tuning ─────────────────────────
del model
torch.cuda.empty_cache()
import gc
gc.collect()
print(f"GPU memory freed: {torch.cuda.memory_allocated() / 1e9:.1f} GB used")

In [ ]:
# ── Prepare training data ──────────────────────────────────────
BEST_PROMPT_TEXT = PROMPTS[BEST_PROMPT]

with open(DATA_DIR / "exams_v" / "train_metadata.json") as f:
    train_data = json.load(f)

train_image_dir = DATA_DIR / "exams_v" / "train" / "images"

print(f"Training samples: {len(train_data)}")
print(f"Prompt: {BEST_PROMPT}")

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

FINETUNE_MODEL = MODEL_8B  # Change to MODEL_4B for Tiny

# ── QLoRA Config ──────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {FINETUNE_MODEL} in 4-bit for QLoRA...")
ft_processor = AutoProcessor.from_pretrained(FINETUNE_MODEL, trust_remote_code=True)
ft_model = Qwen3VLForConditionalGeneration.from_pretrained(
    FINETUNE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

ft_model = prepare_model_for_kbit_training(ft_model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

ft_model = get_peft_model(ft_model, lora_config)
ft_model.print_trainable_parameters()
print(f"GPU memory after QLoRA setup: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ── Build Dataset ─────────────────────────────────────────────
from torch.utils.data import Dataset

class ExamsVDataset(Dataset):
    def __init__(self, metadata, image_dir, processor, prompt_text, max_samples=None):
        self.items = metadata[:max_samples] if max_samples else metadata
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.prompt_text = prompt_text

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        img_path = self.image_dir / f"{item['id']}.png"
        image = Image.open(img_path).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": self.prompt_text},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": item["answer_key"]}],
            },
        ]

        text = self.processor.apply_chat_template(messages, tokenize=False)
        inputs = self.processor(
            text=[text],
            images=[image],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048,
        )

        # Flatten batch dim
        return {k: v.squeeze(0) for k, v in inputs.items()}


max_train = TEST_LIMIT if TEST_MODE else None
train_dataset = ExamsVDataset(
    train_data, train_image_dir, ft_processor, BEST_PROMPT_TEXT, max_samples=max_train
)
print(f"Training dataset: {len(train_dataset)} samples")

In [ ]:
# ── Training ──────────────────────────────────────────────────
ft_output_dir = OUTPUT_DIR / f"qlora_{FINETUNE_MODEL.split('/')[-1]}"

training_args = SFTConfig(
    output_dir=str(ft_output_dir),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    gradient_checkpointing=True,
    report_to="none",
    max_seq_length=2048,
    dataset_text_field=None,
)

trainer = SFTTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_dataset,
)

print(f"Starting QLoRA fine-tuning...")
print(f"  Model: {FINETUNE_MODEL}")
print(f"  Samples: {len(train_dataset)}")
print(f"  Epochs: 3")
print(f"  Effective batch size: {1 * 8}")
print(f"  Output: {ft_output_dir}")

t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 3600
print(f"\nTraining complete in {elapsed:.1f} hours")

# Save LoRA adapter
adapter_path = ft_output_dir / "final_adapter"
ft_model.save_pretrained(str(adapter_path))
ft_processor.save_pretrained(str(adapter_path))
print(f"Adapter saved: {adapter_path}")

In [ ]:
# ── Merge adapter into base model (for vLLM serving) ──────────
from peft import PeftModel

print("Merging LoRA adapter into base model...")

# Free QLoRA model
del ft_model, trainer
torch.cuda.empty_cache()
gc.collect()

# Load base in full precision and merge
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    FINETUNE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

merged_model = PeftModel.from_pretrained(base_model, str(adapter_path))
merged_model = merged_model.merge_and_unload()

merged_path = ft_output_dir / "merged"
merged_model.save_pretrained(str(merged_path))
ft_processor.save_pretrained(str(merged_path))
print(f"Merged model saved: {merged_path}")

del base_model, merged_model
torch.cuda.empty_cache()
gc.collect()

---
## 6. [OPTIONAL] Evaluate Fine-Tuned Model

Skip if you didn't run Section 5. The model from Section 2 is still loaded.

In [ ]:
# ── Load fine-tuned model ──────────────────────────────────────
merged_path = ft_output_dir / "merged"

print(f"Loading fine-tuned model from {merged_path}...")
processor = AutoProcessor.from_pretrained(str(merged_path), trust_remote_code=True)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    str(merged_path),
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation="sdpa",
)
model.eval()
print(f"Loaded. GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ── Evaluate on val subset ────────────────────────────────────
_, ft_acc = run_evaluation(
    val_subset, "validation", BEST_PROMPT, PROMPTS[BEST_PROMPT],
    experiment_name=f"finetuned_{BEST_PROMPT}"
)

print(f"\nZero-shot: {prompt_results[BEST_PROMPT]:.2f}%")
print(f"Fine-tuned: {ft_acc:.2f}%")
print(f"Delta: {ft_acc - prompt_results[BEST_PROMPT]:+.2f}%")

---
## 7. Competition Test Inference (MCQ)

Run on the ACTUAL competition test datasets (separate from EXAMS-V).
- Visual MCQ: `SU-FMI-AI/ImageCLEF-MR2026-MCQ-Visual` (1,117 samples)
- Text MCQ: `SU-FMI-AI/ImageCLEF-MR2026-MCQ-Textual` (1,653 samples)

These have NO answer keys — we submit predictions to the leaderboard.

In [ ]:
SC_K = 3
SC_TEMP = 0.7

def run_mcq_test(dataset_name, experiment_name, k=3, temperature=0.7):
    """Run MCQ inference on competition test data with self-consistency.
    Uses subject-aware + language-matched prompting."""
    meta_path = DATA_DIR / dataset_name / "test_metadata.json"
    image_dir = DATA_DIR / dataset_name / "test" / "images"

    if not meta_path.exists():
        print(f"  {dataset_name}: no test metadata found. Download first!")
        return []

    with open(meta_path) as f:
        data = json.load(f)

    exp_dir = OUTPUT_DIR / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    results_path = exp_dir / "predictions.json"

    results = []
    done_ids = set()
    if results_path.exists():
        with open(results_path) as f:
            results = json.load(f)
        done_ids = {r["question_id"] for r in results}

    items = data if TEST_MODE is False else data[:TEST_LIMIT]
    remaining = [it for it in items if it["question_id"] not in done_ids]

    print(f"  {dataset_name}: {len(remaining)} remaining of {len(items)}")
    print(f"  Self-consistency: k={k}, temp={temperature}")

    for i, item in enumerate(tqdm(remaining, desc=experiment_name)):
        img_path = str(image_dir / f"{item['question_id']}.png")
        if not os.path.exists(img_path):
            print(f"    SKIP {item['question_id']}: image not found")
            continue

        # Build prompt with subject + language awareness
        full_prompt = build_prompt(PROMPTS[BEST_PROMPT], item.get("subject", ""), item.get("language", ""))

        if k > 1:
            answers = []
            for _ in range(k):
                ans, _ = predict_mcq(img_path, full_prompt,
                                     do_sample=True, temperature=temperature)
                answers.append(ans)
            counts = Counter(answers)
            final_answer = counts.most_common(1)[0][0]
            agreement = counts.most_common(1)[0][1] / k
        else:
            final_answer, _ = predict_mcq(img_path, full_prompt)
            counts = {final_answer: 1}
            agreement = 1.0

        results.append({
            "question_id": item["question_id"],
            "answer_key": final_answer,
            "votes": dict(counts) if k > 1 else None,
            "agreement": agreement,
            "language": item.get("language", ""),
            "subject": item.get("subject", ""),
        })

        if (i + 1) % 50 == 0:
            with open(results_path, "w") as f:
                json.dump(results, f, indent=2)

    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)

    print(f"  Saved {len(results)} predictions → {results_path}")
    return results

In [ ]:
# ── Run on COMPETITION test sets ──────────────────────────────
print("MCQ Competition Test Inference")
print("=" * 55)

visual_mcq_results = run_mcq_test(
    "mcq_visual", f"mcq_visual_sc{SC_K}", k=SC_K, temperature=SC_TEMP
)

text_mcq_results = run_mcq_test(
    "mcq_textual", f"mcq_textual_sc{SC_K}", k=SC_K, temperature=SC_TEMP
)

print(f"\nVisual MCQ: {len(visual_mcq_results)} predictions")
print(f"Text MCQ:   {len(text_mcq_results)} predictions")

---
## 8. OpenQA Inference (Competition Test Sets)

- Visual OpenQA: `SU-FMI-AI/ImageCLEF-MR2026-OpenQA-Visual` (439 test)
- Text OpenQA: `SU-FMI-AI/ImageCLEF-MR2026-OpenQA-Textual` (424 test)

In [ ]:
OPENQA_PROMPT = (
    "Look at this exam question image carefully.\n"
    "Read the question and analyze any visual elements.\n"
    "Provide a concise, accurate answer. Be brief — one phrase or short sentence.\n\n"
    "Answer:"
)


def run_openqa_test(dataset_name, experiment_name):
    """Run OpenQA inference on competition test data."""
    meta_path = DATA_DIR / dataset_name / "test_metadata.json"
    image_dir = DATA_DIR / dataset_name / "test" / "images"

    if not meta_path.exists():
        print(f"  {dataset_name}: no test metadata found. Download first!")
        return []

    with open(meta_path) as f:
        data = json.load(f)

    exp_dir = OUTPUT_DIR / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    results_path = exp_dir / "predictions.json"

    results = []
    done_ids = set()
    if results_path.exists():
        with open(results_path) as f:
            results = json.load(f)
        done_ids = {r["question_id"] for r in results}

    items = data if TEST_MODE is False else data[:TEST_LIMIT]
    remaining = [it for it in items if it["question_id"] not in done_ids]

    print(f"  {dataset_name}: {len(remaining)} remaining of {len(items)}")

    for i, item in enumerate(tqdm(remaining, desc=experiment_name)):
        img_path = str(image_dir / f"{item['question_id']}.png")
        if not os.path.exists(img_path):
            results.append({
                "question_id": item["question_id"],
                "answers": [""],
                "language": item.get("language", ""),
            })
            continue

        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{img_path}"},
                {"type": "text", "text": OPENQA_PROMPT},
            ],
        }]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(
            text=[text],
            images=[Image.open(img_path).convert("RGB")],
            return_tensors="pt",
            padding=True,
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)

        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        answer = processor.decode(new_tokens, skip_special_tokens=True).strip()

        results.append({
            "question_id": item["question_id"],
            "answers": [answer],
            "language": item.get("language", ""),
        })

        if (i + 1) % 50 == 0:
            with open(results_path, "w") as f:
                json.dump(results, f, indent=2)

    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)

    print(f"  Saved {len(results)} predictions → {results_path}")
    return results


print("OpenQA Competition Test Inference")
print("=" * 55)
openqa_visual_results = run_openqa_test("openqa_visual", "openqa_visual_test")
openqa_textual_results = run_openqa_test("openqa_textual", "openqa_textual_test")

---
## 9. Package Submissions

In [ ]:
TEAM_NAME = "SRH-ReliableAI"
SUB_DIR = OUTPUT_DIR / "submissions"
SUB_DIR.mkdir(parents=True, exist_ok=True)


def package_mcq_submission(experiment_name, output_name):
    """Format MCQ competition test predictions for submission."""
    results_path = OUTPUT_DIR / experiment_name / "predictions.json"
    if not results_path.exists():
        print(f"  {output_name}: no predictions found at {results_path}")
        return

    with open(results_path) as f:
        preds = json.load(f)

    submission = [
        {"question_id": p["question_id"], "answer_key": p["answer_key"]}
        for p in preds
    ]

    # Validate
    ids = set()
    for item in submission:
        assert item["answer_key"] in {"A", "B", "C", "D", "E"}, f"Invalid: {item['answer_key']}"
        assert item["question_id"] not in ids, f"Duplicate: {item['question_id']}"
        ids.add(item["question_id"])

    out_path = SUB_DIR / f"{output_name}.json"
    with open(out_path, "w") as f:
        json.dump(submission, f, indent=2, ensure_ascii=False)
    print(f"  {output_name}: {len(submission)} predictions -> {out_path}")


def package_openqa_submission(experiment_name, output_name):
    """Format OpenQA competition test predictions for submission."""
    results_path = OUTPUT_DIR / experiment_name / "predictions.json"
    if not results_path.exists():
        print(f"  {output_name}: no predictions found at {results_path}")
        return

    with open(results_path) as f:
        preds = json.load(f)

    submission = [
        {
            "question_id": p["question_id"],
            "answers": p["answers"],
            "language": p.get("language", ""),
        }
        for p in preds
    ]

    out_path = SUB_DIR / f"{output_name}.json"
    with open(out_path, "w") as f:
        json.dump(submission, f, indent=2, ensure_ascii=False)
    print(f"  {output_name}: {len(submission)} predictions -> {out_path}")


print("Packaging Submissions")
print("=" * 55)

# MCQ submissions
package_mcq_submission(f"mcq_visual_sc{SC_K}", "visual_mcq_normal")
package_mcq_submission(f"mcq_textual_sc{SC_K}", "text_mcq_normal")

# OpenQA submissions
package_openqa_submission("openqa_visual_test", "visual_openqa_normal")
package_openqa_submission("openqa_textual_test", "text_openqa_normal")

print(f"\nAll submissions in: {SUB_DIR}")
print()
print("Upload to leaderboard:")
print("  Visual MCQ:   https://ai4media-bench.aimultimedialab.ro/competitions/12/")
print("  Text MCQ:     https://ai4media-bench.aimultimedialab.ro/competitions/16/")
print("  Visual OpenQA: check AI4Media-Bench for link")
print("  Text OpenQA:   check AI4Media-Bench for link")

---
## 10. Analysis

In [ ]:
def analyze_results(experiment_name):
    """Detailed per-language and per-subject analysis."""
    results_path = OUTPUT_DIR / experiment_name / "predictions.json"
    if not results_path.exists():
        print(f"No results found for {experiment_name}")
        return

    with open(results_path) as f:
        results = json.load(f)

    if not results or "correct" not in results[0]:
        print(f"{experiment_name}: no gold labels available")
        return

    correct = sum(1 for r in results if r["correct"])
    total = len(results)
    print(f"\n{'='*60}")
    print(f"  {experiment_name}: {correct}/{total} = {correct/total*100:.2f}%")
    print(f"{'='*60}")

    # Per language
    lang_stats = defaultdict(lambda: {"c": 0, "t": 0})
    for r in results:
        lang_stats[r["language"]]["t"] += 1
        if r["correct"]:
            lang_stats[r["language"]]["c"] += 1

    print(f"\nPer-language:")
    for lang in sorted(lang_stats, key=lambda l: -lang_stats[l]["t"]):
        s = lang_stats[lang]
        print(f"  {lang:<15} {s['c']:>4}/{s['t']:<4} = {s['c']/s['t']*100:5.1f}%")

    # Per subject (top 10)
    subj_stats = defaultdict(lambda: {"c": 0, "t": 0})
    for r in results:
        subj_stats[r.get("subject", "?")]["t"] += 1
        if r["correct"]:
            subj_stats[r.get("subject", "?")]["c"] += 1

    print(f"\nPer-subject (top 10):")
    for subj in sorted(subj_stats, key=lambda s: -subj_stats[s]["t"])[:10]:
        s = subj_stats[subj]
        print(f"  {subj:<25} {s['c']:>4}/{s['t']:<4} = {s['c']/s['t']*100:5.1f}%")

    # Per type
    type_stats = defaultdict(lambda: {"c": 0, "t": 0})
    for r in results:
        type_stats[r.get("type", "?")]["t"] += 1
        if r["correct"]:
            type_stats[r.get("type", "?")]["c"] += 1

    print(f"\nPer-type:")
    for t in sorted(type_stats):
        s = type_stats[t]
        print(f"  {t:<15} {s['c']:>4}/{s['t']:<4} = {s['c']/s['t']*100:5.1f}%")


# Analyze all experiments
for exp in sorted(OUTPUT_DIR.iterdir()):
    if exp.is_dir() and (exp / "predictions.json").exists():
        analyze_results(exp.name)

---
## Checklist

### Before full run:
- [ ] Set `TEST_MODE = False`
- [ ] All data downloaded
- [ ] Model loads successfully
- [ ] Test run gives reasonable answers
- [ ] Registered on AI4Media-Bench (all 4 tracks)

### After baseline:
- [ ] Best prompt identified from ablation
- [ ] Zero-shot accuracy established

### After fine-tuning:
- [ ] QLoRA training completed without OOM
- [ ] Fine-tuned model shows improvement over zero-shot
- [ ] Adapter merged and saved

### After self-consistency:
- [ ] SC k=3 shows improvement over single-pass
- [ ] Agreement analysis looks reasonable

### Submissions:
- [ ] Visual MCQ JSON formatted and validated
- [ ] Text MCQ JSON formatted and validated
- [ ] Visual OpenQA JSON formatted
- [ ] Text OpenQA JSON formatted
- [ ] Uploaded to AI4Media-Bench
- [ ] Check leaderboard → iterate
- [ ] Final submission by May 7

### For Tiny category:
- [ ] Change `ACTIVE_MODEL` and `FINETUNE_MODEL` to `MODEL_4B`
- [ ] Re-run sections 5-9
- [ ] Submit separately as Tiny